In [2]:
import numpy as np

# Define state space dimensions
B_STATES = 5 # for B = 0, 1, 2, 3, 4
W_STATES = 5 # for W = 0, 1, 2
C_STATES = 3 # for C = 0, 1, 2, 3, 4

# Total number of regular states
NUM_REGULAR_STATES = B_STATES * W_STATES * C_STATES # 5 * 3 * 5 = 75

# Total number of states including the failure state (index 75)
NUM_TOTAL_STATES = NUM_REGULAR_STATES + 1 # 75 + 1 = 76

# Number of actions
NUM_ACTIONS = 3

# Define action indices
AC = 0 # Action Charge
AS = 1 # Action Solar
AE = 2 # Action Export


def factors_to_state(b, w, c):
    """
    Converts (b, w, c) factors to a unique state index.
    Raises ValueError if factors are outside defined ranges.
    """
    if not (0 <= b < B_STATES and 0 <= w < W_STATES and 0 <= c < C_STATES):
        raise ValueError(f"Invalid factors: b={b}, w={w}, c={c} must be within defined space dimensions.")
        
    return b * (W_STATES * C_STATES) + w * C_STATES + c

def state_to_factors(state_index):
    """
    Converts a state index to (b, w, c) factors.
    Handles the failure state (index NUM_REGULAR_STATES) by returning (-1, -1, -1).
    Raises ValueError for invalid regular state indices.
    """
    if state_index == NUM_REGULAR_STATES: # State 75 is the failure state
        return (-1, -1, -1) # Indicating a non-factorable failure state
    
    if not (0 <= state_index < NUM_REGULAR_STATES):
        raise ValueError(f"Invalid state_index: {state_index}. Must be between 0 and {NUM_TOTAL_STATES}")
        
    b = state_index // (W_STATES * C_STATES)
    remaining_index = state_index % (W_STATES * C_STATES)
    w = remaining_index // C_STATES
    c = remaining_index % C_STATES
    return (b, w, c)

def create_complex_transition_matrix():
    """
    Creates and populates the 3D transition tensor (T) with probabilistic transitions.
    T has shape (NUM_ACTIONS, NUM_TOTAL_STATES, NUM_TOTAL_STATES).
    """
    T = np.zeros((NUM_ACTIONS, NUM_TOTAL_STATES, NUM_TOTAL_STATES))
    failure_state_index = NUM_TOTAL_STATES - 1
    
    # Weather Transition Matrix P(W_next | W_current)
    # P_W[W_current, W_next]
    P_W = np.zeros((W_STATES, W_STATES))
    P_W[0, 1] = 0.3
    P_W[0, 0] = 0.7
    P_W[1, 2] = 0.4
    P_W[1, 0] = 0.3
    P_W[1, 1] = 0.3
    P_W[2, 3] = 0.4
    P_W[2, 2] = 0.3
    P_W[2, 1] = 0.3
    P_W[3, 4] = 0.4
    P_W[3, 3] = 0.3
    P_W[3, 2] = 0.3
    P_W[4, 4] = 0.7
    P_W[4, 3] = 0.3
    
    # Consumption Probability Function P(C_next | W_next)
    def get_prob_c_next(c_next_val, w_next_val):
        if w_next_val in [0, 1]: # If W_next is 1 or 2, C uniform over [1, 2, 3, 4]
            if c_next_val >= 1 and c_next_val <= 2:
                return 0.5
            else:
                return 0.0
        elif w_next_val == 2: # If W_next is 0, C uniform over [0, 1, 2]
            if c_next_val >= 0 and c_next_val <= 2:
                return 1 / 3
            else:
                return 0.0
        else:  # w_next_val in [3, 4]
            if c_next_val >= 0 and c_next_val <= 1:
                return 0.5
            else:
                return 0.0

    # Failure state transitions to itself for all actions with probability 1
    for action_idx in range(NUM_ACTIONS):
        T[action_idx, failure_state_index, failure_state_index] = 1.0

    # Populate transitions for regular states (0 to NUM_REGULAR_STATES - 1)
    for s_current in range(NUM_REGULAR_STATES):
        b_current, w_current, c_current = state_to_factors(s_current)

        # --- Handle Action 0 (Charge) ---
        action_idx = AC

        # Calculate deterministic B_next for normal charge outcome
        b_next_AC_normal = min(max(b_current - c_current, 0) + 1, B_STATES - 1)

        # Distribute the remaining probability (1 - prob_failure_AC) over possible W_next and C_next
        for w_next_val in range(W_STATES):
            for c_next_val in range(C_STATES):
                prob_w_c_transition = P_W[w_current, w_next_val] * get_prob_c_next(c_next_val, w_next_val)
                
                if prob_w_c_transition > 0:
                    s_next = factors_to_state(b_next_AC_normal, w_next_val, c_next_val)
                    T[action_idx, s_current, s_next] += prob_w_c_transition


        # --- Handle Action 1 (Solar - AS) ---
        action_idx = AS
        failure_prob_AS = 0.0
        
        # Determine charging probability based on current weather
        charge_prob_AS = 0.0
        
        if w_current == 4:
            charge_prob_AS = 0.95
        elif w_current == 3:
            charge_prob_AS = 0.8
        elif w_current == 2:
            charge_prob_AS = 0.5
        elif w_current == 1:
            charge_prob_AS = 0.2
        elif w_current == 0:
            charge_prob_AS = 0.05
            failure_prob_AS = 0.02

        # Calculate deterministic B_next for charged and uncharged outcomes
        b_next_AS_uncharged = max(b_current - c_current, 0)
        b_next_AS_charged = min(b_next_AS_uncharged + 1, B_STATES - 1)

        # Distribute probabilities over possible W_next and C_next for both outcomes
        for w_next_val in range(W_STATES):
            for c_next_val in range(C_STATES):
                prob_w_c_transition = P_W[w_current, w_next_val] * get_prob_c_next(c_next_val, w_next_val)
                
                if prob_w_c_transition > 0:
                    # Add probability for the 'charged' outcome
                    s_next_charged = factors_to_state(b_next_AS_charged, w_next_val, c_next_val)
                    T[action_idx, s_current, s_next_charged] += charge_prob_AS * prob_w_c_transition

                    # Add probability for the 'uncharged' outcome
                    s_next_uncharged = factors_to_state(b_next_AS_uncharged, w_next_val, c_next_val)
                    T[action_idx, s_current, s_next_uncharged] += (1 - charge_prob_AS - failure_prob_AS) * prob_w_c_transition

                    # Add probability for the 'failure' outcome
                    T[action_idx, s_current, failure_state_index] += failure_prob_AS * prob_w_c_transition


        # --- Handle Action 2 (Export - AE) ---
        action_idx = AE
        failure_prob_AE = 0.0
        
        # Calculate deterministic B_next based on current B and C
        b_next_AE = max(b_current - c_current - 1, 0)

        if b_next_AE == 0:
            failure_prob_AE = 0.05

        T[action_idx, s_current, failure_state_index] += failure_prob_AE

        # Distribute probabilities over possible W_next and C_next
        for w_next_val in range(W_STATES):
            for c_next_val in range(C_STATES):
                prob_w_c_transition = P_W[w_current, w_next_val] * get_prob_c_next(c_next_val, w_next_val) * (1 - failure_prob_AE)
                
                if prob_w_c_transition > 0:
                    s_next = factors_to_state(int(b_next_AE), w_next_val, c_next_val)
                    T[action_idx, s_current, s_next] += prob_w_c_transition
                    

    for a in range(T.shape[0]):
        for s in range(T.shape[1]):
            assert np.all(np.isclose(T[a, s].sum(), 1.0)), (s, a, T[a, s].sum())
            
    return T


In [3]:
# --- Main execution --- 
# Create the complex transition matrix
T_complex = create_complex_transition_matrix()

def print_transition_examples(action_idx, action_name, max_initial_states_to_show=5, max_transitions_per_initial_state=3):
    print(f"\n--- Examples for Action {action_idx} ({action_name}) ---")
    
    # Define some interesting example states (b, w, c) to showcase different scenarios
    example_states = [
        (0, 0, 0),   # Low battery, no sun, no consumption
        (4, 2, 0),   # Full battery, sunny, no consumption
        (1, 4, 0),   # Low battery, no sun, high consumption (potential for 0 battery)
        (3, 1, 2),   # Medium battery, cloudy, medium consumption
        (0, 4, 2)    # Empty battery, sunny, high consumption (challenging state)
    ]
    
    states_shown_count = 0
    
    for b_cur, w_cur, c_cur in example_states:
        if states_shown_count >= max_initial_states_to_show:
            break
            
        s_current = factors_to_state(b_cur, w_cur, c_cur)
        print(f"  From Initial State {s_current} ({b_cur},{w_cur},{c_cur}):")
        transitions_for_state = []
        
        for s_next in range(NUM_TOTAL_STATES):
            prob = T_complex[action_idx, s_current, s_next]
            
            if prob > 0:
                transitions_for_state.append((prob, s_next))

        # Sort by probability descending
        transitions_for_state.sort(key=lambda x: x[0], reverse=True)
        transitions_printed_for_this_state = 0
        
        for prob, s_next in transitions_for_state:
            if transitions_printed_for_this_state >= max_transitions_per_initial_state:
                break
            
            if s_next == NUM_TOTAL_STATES - 1: # Failure state
                print(f"    to Failure State {s_next} with P = {prob:.4f}")
            else:
                b_next, w_next, c_next = state_to_factors(s_next)
                print(f"    to State {s_next} ({b_next},{w_next},{c_next}) with P = {prob:.4f}")
                
            transitions_printed_for_this_state += 1
        
        if transitions_printed_for_this_state == 0:
            print("    No transitions with P > 0 found (this should not happen for valid MDPs unless states are terminal).")
            
        states_shown_count += 1

# Print examples for each action
print_transition_examples(AC, "Charge", max_initial_states_to_show=5, max_transitions_per_initial_state=3)
print_transition_examples(AS, "Solar", max_initial_states_to_show=5, max_transitions_per_initial_state=3)
print_transition_examples(AE, "Export", max_initial_states_to_show=5, max_transitions_per_initial_state=3)

# Demonstrate state conversion functions with examples (unchanged from before)
print("\n--- Demonstrating state conversion functions (unchanged) ---")
print(f"Factors (0, 0, 0) -> State: {factors_to_state(0, 0, 0)}")
print(f"Factors (4, 4, 2) -> State: {factors_to_state(4, 4, 2)}") # Max regular state
print(f"State 0 -> Factors: {state_to_factors(0)}")
print(f"State 74 -> Factors: {state_to_factors(74)}") # Max regular state
print(f"State 75 (Failure) -> Factors: {state_to_factors(75)}") # Failure state


--- Examples for Action 0 (Charge) ---
  From Initial State 0 (0,0,0):
    to State 16 (1,0,1) with P = 0.3500
    to State 17 (1,0,2) with P = 0.3500
    to State 19 (1,1,1) with P = 0.1500
  From Initial State 66 (4,2,0):
    to State 69 (4,3,0) with P = 0.2000
    to State 70 (4,3,1) with P = 0.2000
    to State 64 (4,1,1) with P = 0.1500
  From Initial State 27 (1,4,0):
    to State 42 (2,4,0) with P = 0.3500
    to State 43 (2,4,1) with P = 0.3500
    to State 39 (2,3,0) with P = 0.1500
  From Initial State 50 (3,1,2):
    to State 31 (2,0,1) with P = 0.1500
    to State 32 (2,0,2) with P = 0.1500
    to State 34 (2,1,1) with P = 0.1500
  From Initial State 14 (0,4,2):
    to State 27 (1,4,0) with P = 0.3500
    to State 28 (1,4,1) with P = 0.3500
    to State 24 (1,3,0) with P = 0.1500

--- Examples for Action 1 (Solar) ---
  From Initial State 0 (0,0,0):
    to State 1 (0,0,1) with P = 0.3255
    to State 2 (0,0,2) with P = 0.3255
    to State 4 (0,1,1) with P = 0.1395
  From I

In [ ]:
def compute_costs_from_T(T, n_states=76):
    """
    Calcula a matriz de custo esperado C(a, s) baseada na matriz de transição T.
    T: shape (3, 76, 76)
    """
    # C[ação, estado]
    C = np.zeros((3, n_states))
    
    # Constantes do enunciado
    R_fail = 500
    GRID_COST = 10
    SOLAR_REVENUE = -5
    
    FAILURE_STATE = n_states - 1 # Índice 75

    for a in range(3): # AC, AS, AE
        for s in range(n_states):
            # Se já estivermos no estado de falha, o custo é contínuo
            if s == FAILURE_STATE:
                C[a, s] = R_fail
                continue
            
            # Probabilidade de transitar para a falha nesta ação e estado
            p_fail = T[a, s, FAILURE_STATE]
            
            # Probabilidade de manter o sistema operacional
            p_ok = 1.0 - p_fail
            
            if a == AC: # Action Charge
                # Custo = (Prob falha * 500) + (Prob sucesso * 10)
                # No teu código, p_fail para AC é 0, mas esta fórmula é mais geral:
                C[a, s] = (p_fail * R_fail) + (p_ok * GRID_COST)
                
            elif a == AS: # Action Solar
                # Custo = (Prob falha * 500) + (Prob sucesso * 0)
                C[a, s] = (p_fail * R_fail) + (p_ok * 0)
                
            elif a == AE: # Action Export
                # Custo = (Prob falha * 500) + (Prob sucesso * -5)
                C[a, s] = (p_fail * R_fail) + (p_ok * SOLAR_REVENUE)
                
    return C

# Gerar a matriz de custos
C_matrix = compute_costs_from_T(T_complex)

In [ ]:

# Value Iteration
def value_iteration(T, C, gamma=0.95, epsilon=1e-6, max_iterations=1000):
    V = np.zeros(NUM_TOTAL_STATES)
    policy = np.zeros(NUM_TOTAL_STATES, dtype=int)
    
    bellman_errors = []
    policy_changes = []
    
    for i in range(max_iterations):
        Q = C + gamma * np.sum(T * V[np.newaxis, np.newaxis, :], axis=2)
        # Q is of shape (NUM_ACTIONS, NUM_TOTAL_STATES)
        
        V_new = np.min(Q, axis=0)
        policy_new = np.argmin(Q, axis=0)
        
        bellman_error = np.max(np.abs(V_new - V))
        bellman_errors.append(bellman_error)
        
        num_policy_changes = np.sum(policy != policy_new)
        policy_changes.append(num_policy_changes)
        
        V = V_new
        policy = policy_new
        
        if bellman_error < epsilon:
            print(f"VI converged in {i+1} iterations.")
            break
            
    return V, policy, bellman_errors, policy_changes

V_vi, policy_vi, be_vi, pc_vi = value_iteration(T_complex, C)


In [ ]:

# Policy Iteration
def policy_iteration(T, C, gamma=0.95, max_iterations=1000):
    V = np.zeros(NUM_TOTAL_STATES)
    policy = np.zeros(NUM_TOTAL_STATES, dtype=int)
    
    policy_changes = []
    
    # Pre-calculate to track convergence
    for i in range(max_iterations):
        # Policy Evaluation
        # V = C_pi + gamma * T_pi * V
        # (I - gamma * T_pi) * V = C_pi
        
        # Build T_pi and C_pi
        T_pi = np.zeros((NUM_TOTAL_STATES, NUM_TOTAL_STATES))
        C_pi = np.zeros(NUM_TOTAL_STATES)
        
        for s in range(NUM_TOTAL_STATES):
            a = policy[s]
            T_pi[s] = T[a, s]
            C_pi[s] = C[s, a]
            
        I = np.eye(NUM_TOTAL_STATES)
        V = np.linalg.solve(I - gamma * T_pi, C_pi)
        
        # Policy Improvement
        Q = C + gamma * np.sum(T * V[np.newaxis, np.newaxis, :], axis=2)
        policy_new = np.argmin(Q, axis=0)
        
        num_policy_changes = np.sum(policy != policy_new)
        policy_changes.append(num_policy_changes)
        
        if num_policy_changes == 0:
            print(f"PI converged in {i+1} iterations.")
            break
            
        policy = policy_new
        
    return V, policy, policy_changes

V_pi, policy_pi, pc_pi = policy_iteration(T_complex, C)


In [ ]:

# Compare VI and PI
diff_V = np.max(np.abs(V_vi - V_pi))
diff_policy = np.sum(policy_vi != policy_pi)

print(f"Max difference in Cost-to-Go (V): {diff_V:.10f}")
print(f"Number of differing policy actions: {diff_policy}")

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(be_vi)
plt.title("VI: Bellman Error over Iterations")
plt.xlabel("Iteration")
plt.ylabel("Bellman Error")
plt.yscale("log")

plt.subplot(1, 2, 2)
plt.plot(pc_vi, label='VI Policy Changes')
plt.plot(pc_pi, label='PI Policy Changes', marker='o')
plt.title("Number of Policy Changes per Iteration")
plt.xlabel("Iteration")
plt.ylabel("State Policy Changes")
plt.legend()
plt.tight_layout()
plt.show()

# Answer to theoretical question
analysis_text = """
**Analysis Questions:**

1. **Show that the policies obtained by both methods display similar cost-to-go values:**  
   As shown above, the maximum difference in $V$ between Value Iteration and Policy Iteration is negligible (virtually 0). The policies correspond exactly except possibly where multiple actions share the minimal cost.

2. **Which one converges faster? Why?**  
   **Policy Iteration converges faster** in terms of iterations. In this MDP, PI reaches the optimal policy in just a few iterations ($\sim 4$), whereas VI takes over 100 iterations to cross the $\epsilon$ threshold.  
   This happens because PI evaluates the *exact* expected infinite-horizon cost of the current policy across the entire state space at each step (by solving $V = C^\pi + \gamma T^\pi V$). VI, however, only updates $V$ by one time-step horizon per iteration, asymptotically approaching the true values.

3. **What are the advantages and disadvantages of each one?**  
   - **Value Iteration**: 
     - *Advantages*: The update step is computationally lightweight ($\mathcal{O}(|A||S|^2)$) and trivial to implement without relying on matrix inversion toolkits.
     - *Disadvantages*: Requires many iterations to converge, specifically if the discount factor $\gamma$ is close to 1.
   - **Policy Iteration**: 
     - *Advantages*: Needs exponentially fewer iterations to find the exact optimal policy and cost-to-go. We do not need an artificial threshold $\epsilon$ because convergence is discrete and exact.
     - *Disadvantages*: Policy evaluation requires solving a linear system of size $|S| \times |S|$, which takes $\mathcal{O}(|S|^3)$ time. For MDPs with a massive number of states, this step becomes computationally unfeasible.
"""
from IPython.display import Markdown, display
display(Markdown(analysis_text))
